# Defactify — Detekcja AI-Generated Images (v3 — naprawiony)

### Co było nie tak w poprzednich wersjach?

| Problem | Skutek | Naprawa |
|---------|--------|---------|
| **`/255.0` normalizacja** | EfficientNet ma wbudowaną normalizację — podwójne skalowanie dawało wartości ~0 | Podajemy surowe piksele [0, 255] |
| **Backbone w 100% zamrożony** | Cechy ImageNet rozpoznają obiekty, NIE artefakty AI. Zamrożone = bezużyteczne cechy | Odmrażamy top ~30% warstw od startu |
| **BatchNorm po zamrożonym backbone** | BN w trybie inference używa statystyk ImageNet, nie naszych danych | Usunięty + backbone trenowany = BN się adaptuje |
| **LR=1e-3 przy transfer learning** | Za agresywne — niszczy pretrenowane wagi | LR=1e-4 z warmup |

### Pipeline
1. Logowanie HuggingFace + pobranie datasetu
2. Diagnostyka danych (weryfikacja zakresów, etykiet)
3. Preprocessing + bezpieczna augmentacja
4. Model z częściowo odmrożonym backbone
5. Trening z callbackami + zapis


In [ ]:
# !pip install datasets pillow matplotlib tensorflow scikit-learn huggingface_hub

In [ ]:
# Logowanie do HuggingFace
!huggingface-cli login

# Alternatywy:
# from huggingface_hub import notebook_login; notebook_login()
# from huggingface_hub import login; login(token="hf_TwojToken")

In [ ]:
from datasets import load_dataset
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf

print(f"TensorFlow: {tf.__version__}")
print(f"GPU: {tf.config.list_physical_devices('GPU')}")

ds = load_dataset("Rajarshi-Roy-research/Defactify_Image_Dataset")
train = ds['train']

print(f"\nRozmiar:  {len(train)} obrazów")
print(f"Kolumny:  {train.column_names}")

In [ ]:
LABEL_A_MAP = {0: "Real", 1: "AI-Generated"}
LABEL_B_MAP = {
    0: "Real", 1: "Stable Diffusion 2.1", 2: "Stable Diffusion XL",
    3: "Stable Diffusion 3", 4: "DALL-E 3", 5: "Midjourney 6"
}

## 1. Diagnostyka — zanim wrzucimy dane do modelu, sprawdźmy je

In [ ]:
# ─── DIAGNOSTYKA: Czy dane są OK? ───
# Sprawdzamy zakresy pikseli, kształty, rozkład etykiet

from collections import Counter
import math

# Losowe 5 próbek
print("=== Próbki danych ===")
for i in [0, 100, 7000, 15000, 30000]:
    sample = train[i]
    img = np.array(sample['Image'].convert('RGB'))
    label_a = sample['Label_A']
    label_b = sample['Label_B']
    print(f"  [{i}] shape={img.shape}, dtype={img.dtype}, "
          f"range=[{img.min()}, {img.max()}], "
          f"Label_A={label_a} ({LABEL_A_MAP[label_a]}), "
          f"Label_B={label_b} ({LABEL_B_MAP[label_b]})")

# Rozkład klas
label_a_counts = Counter(train['Label_A'])
print(f"\n=== Rozkład Label_A ===")
for k, v in sorted(label_a_counts.items()):
    print(f"  {LABEL_A_MAP[k]:15s}: {v:6d}  ({v/len(train)*100:.1f}%)")

# Sprawdzenie: czy Label_A=0 to ZAWSZE Label_B=0?
mismatches = 0
for i in range(min(1000, len(train))):
    s = train[i]
    if s['Label_A'] == 0 and s['Label_B'] != 0:
        mismatches += 1
    if s['Label_A'] == 1 and s['Label_B'] == 0:
        mismatches += 1
if mismatches == 0:
    print("\n✅ Etykiety Label_A i Label_B są spójne (sprawdzono 1000 próbek)")
else:
    print(f"\n⚠️ Znaleziono {mismatches} niespójnych etykiet!")

In [ ]:
# 10 przykładów — 5 Real + 5 AI
fig, axes = plt.subplots(2, 5, figsize=(20, 10))
axes = axes.flatten()

# Znajdź indeksy Real i AI
real_indices = [i for i in range(min(5000, len(train))) if train[i]['Label_A'] == 0][:5]
ai_indices = [i for i in range(min(5000, len(train))) if train[i]['Label_A'] == 1][:5]
show_indices = real_indices + ai_indices

for i, idx in enumerate(show_indices):
    sample = train[idx]
    ax = axes[i]
    ax.imshow(sample['Image'])
    la, lb = sample['Label_A'], sample['Label_B']
    ax.set_title(
        f"{LABEL_A_MAP[la]} | {LABEL_B_MAP[lb]}",
        fontsize=11, fontweight='bold',
        color='green' if la == 0 else 'red'
    )
    cap = sample['Caption']
    ax.set_xlabel(cap[:60] + "..." if len(cap) > 60 else cap, fontsize=8)
    ax.set_xticks([]); ax.set_yticks([])

plt.suptitle("Góra: Real | Dół: AI-Generated", fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import train_test_split

labels_all = np.array(train['Label_A'])

class_weights_arr = compute_class_weight(
    class_weight='balanced',
    classes=np.array([0, 1]),
    y=labels_all
)
class_weight_dict = {0: class_weights_arr[0], 1: class_weights_arr[1]}
print(f"Class weights: Real={class_weight_dict[0]:.3f}, AI={class_weight_dict[1]:.3f}")
print(f"  → Real ma {class_weight_dict[0]/class_weight_dict[1]:.1f}x większą wagę w loss")

In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE
SEED = 42

indices = list(range(len(train)))
train_idx, val_idx = train_test_split(
    indices, test_size=0.2, random_state=SEED, stratify=labels_all
)

STEPS_PER_EPOCH = math.ceil(len(train_idx) / BATCH_SIZE)
VALIDATION_STEPS = math.ceil(len(val_idx) / BATCH_SIZE)

print(f"Train: {len(train_idx)} | Val: {len(val_idx)}")
print(f"Steps per epoch: {STEPS_PER_EPOCH} | Validation steps: {VALIDATION_STEPS}")

## 2. Data Pipeline

**Kluczowa zmiana:** EfficientNetB0 w TF/Keras ma wbudowane warstwy `Rescaling` i `Normalization`.
Podajemy surowe piksele **[0, 255]** — model sam normalizuje. Żadnego `/255.0` ani `preprocess_input`!


In [ ]:
# ─── Preprocessing: TYLKO resize, BEZ normalizacji ───
# EfficientNetB0 ma wbudowaną normalizację — podajemy [0, 255]

def preprocess_image(image_pil, label):
    """PIL Image → float32 tensor [0, 255] o rozmiarze (224,224,3)."""
    img = image_pil.convert('RGB').resize((IMG_SIZE, IMG_SIZE))
    img = np.array(img, dtype=np.float32)  # [0, 255] float32
    return img, label

# ─── Bezpieczna augmentacja ───
# Operuje na [0, 255] — subtelna, nie niszczy artefaktów AI
def safe_augment(image, label):
    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_brightness(image, max_delta=15.0)  # ±15 z 255
    image = tf.image.random_contrast(image, lower=0.9, upper=1.1)
    image = tf.clip_by_value(image, 0.0, 255.0)
    return image, label

# ─── Generator ───
def make_generator(indices_list, dataset, augment=False):
    def gen():
        shuffled = np.random.permutation(indices_list)
        for idx in shuffled:
            sample = dataset[int(idx)]
            img, label = preprocess_image(sample['Image'], sample['Label_A'])
            yield img, label

    ds = tf.data.Dataset.from_generator(
        gen,
        output_signature=(
            tf.TensorSpec(shape=(IMG_SIZE, IMG_SIZE, 3), dtype=tf.float32),
            tf.TensorSpec(shape=(), dtype=tf.int32),
        )
    )
    if augment:
        ds = ds.map(safe_augment, num_parallel_calls=AUTOTUNE)

    ds = ds.batch(BATCH_SIZE).repeat().prefetch(AUTOTUNE)
    return ds

train_ds = make_generator(train_idx, train, augment=True)
val_ds = make_generator(val_idx, train, augment=False)

print("Datasety utworzone ✅")

In [ ]:
# ─── WERYFIKACJA: Sprawdź co model faktycznie dostaje ───
for images, labels in train_ds.take(1):
    print(f"Batch shape:  {images.shape}")
    print(f"Dtype:        {images.dtype}")
    print(f"Pixel range:  [{images.numpy().min():.1f}, {images.numpy().max():.1f}]")
    print(f"Mean pixel:   {images.numpy().mean():.1f}")
    print(f"Labels:       {labels.numpy()[:16]}")
    print(f"Label dist:   0={np.sum(labels.numpy()==0)}, 1={np.sum(labels.numpy()==1)}")

    # Sprawdź czy zakres jest [0, 255]
    pmin, pmax = images.numpy().min(), images.numpy().max()
    if pmax <= 1.0:
        print("\n❌ BŁĄD: Piksele w zakresie [0,1] — EfficientNet oczekuje [0,255]!")
    elif pmax > 1.0 and pmax <= 255.0:
        print("\n✅ Zakres pikseli OK — [0, 255] dla EfficientNet")

    # Wizualizacja pierwszych 4 obrazów z batcha
    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    for i in range(4):
        axes[i].imshow(images[i].numpy().astype(np.uint8))
        axes[i].set_title(f"Label: {labels[i].numpy()} ({LABEL_A_MAP[labels[i].numpy()]})",
                         color='green' if labels[i].numpy() == 0 else 'red',
                         fontweight='bold')
        axes[i].set_xticks([]); axes[i].set_yticks([])
    plt.suptitle("Dane wejściowe do modelu (tak je widzi sieć)", fontweight='bold')
    plt.tight_layout()
    plt.show()
    break

## 3. Model — EfficientNetB0 z CZĘŚCIOWYM odmrożeniem

**Dlaczego nie zamrażamy całego backbone:**

ImageNet uczy sieci rozpoznawać *obiekty* (kot, samochód, most). Ale detekcja AI-generated images
wymaga rozpoznawania *artefaktów generatora* — subtelnych wzorców w teksturach, częstotliwościach,
kompresji. Te cechy NIE istnieją w zamrożonych wagach ImageNet.

**Strategia:** Odmrażamy ostatnie ~30% warstw EfficientNet (bloki 5-7).
Niższe warstwy (filtry krawędzi, tekstury) zostawiamy zamrożone — te się przydają.
Wyższe warstwy adaptujemy do naszego zadania.


In [ ]:
from tensorflow.keras import layers, models, callbacks
from tensorflow.keras.applications import EfficientNetB0
import os

MODEL_DIR = "saved_models"
os.makedirs(MODEL_DIR, exist_ok=True)

CHECKPOINT_PATH = os.path.join(MODEL_DIR, "best_model.keras")
FINAL_MODEL_PATH = os.path.join(MODEL_DIR, "final_model.keras")
WEIGHTS_PATH = os.path.join(MODEL_DIR, "model_weights.weights.h5")

def build_model(unfreeze_from=100):
    """
    EfficientNetB0 z odmrożonymi warstwami od indeksu `unfreeze_from`.

    EfficientNetB0 ma ~236 warstw:
      - unfreeze_from=0   → wszystko odmrożone (pełny fine-tuning)
      - unfreeze_from=100 → ~60% zamrożone, 40% trenowane (zalecane na start)
      - unfreeze_from=200 → ~85% zamrożone, 15% trenowane (lekki fine-tuning)
      - unfreeze_from=None → backbone w pełni zamrożony (NIE działa dla tego zadania!)
    """
    base_model = EfficientNetB0(
        include_top=False,
        weights='imagenet',
        input_shape=(IMG_SIZE, IMG_SIZE, 3)
    )

    # Odmrażanie warstw
    if unfreeze_from is None:
        base_model.trainable = False
        print("Backbone: w pełni zamrożony")
    else:
        base_model.trainable = True
        for layer in base_model.layers[:unfreeze_from]:
            layer.trainable = False
        frozen = sum(1 for l in base_model.layers if not l.trainable)
        trainable = sum(1 for l in base_model.layers if l.trainable)
        print(f"Backbone: {frozen} zamrożonych + {trainable} trenowalnych "
              f"(z {len(base_model.layers)} łącznie)")

    # Klasyfikator — prostszy, bez BatchNorm (unikamy konfliktu z BN w backbone)
    inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = base_model(inputs, training=True)  # training=True → BN aktualizuje statystyki
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(1, activation='sigmoid')(x)

    model = models.Model(inputs, outputs)
    return model

model = build_model(unfreeze_from=100)

# Podsumowanie
total_params = model.count_params()
trainable_params = sum(tf.keras.backend.count_params(w) for w in model.trainable_weights)
frozen_params = total_params - trainable_params
print(f"\nParametry: {total_params:,} łącznie")
print(f"  Trenowalne: {trainable_params:,} ({trainable_params/total_params*100:.1f}%)")
print(f"  Zamrożone:  {frozen_params:,} ({frozen_params/total_params*100:.1f}%)")

In [ ]:
# ─── Kompilacja ───
# Mniejszy LR niż poprzednio — fine-tuning wymaga ostrożności

LEARNING_RATE = 1e-4  # 10x mniejszy niż domyślny Adam

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss='binary_crossentropy',
    metrics=[
        'accuracy',
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall'),
        tf.keras.metrics.AUC(name='auc')
    ]
)

# ─── Callbacks ───
my_callbacks = [
    callbacks.ModelCheckpoint(
        filepath=CHECKPOINT_PATH,
        monitor='val_auc',
        mode='max',
        save_best_only=True,
        verbose=1
    ),
    callbacks.EarlyStopping(
        monitor='val_auc',
        mode='max',
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=2,
        min_lr=1e-7,
        verbose=1
    ),
]

print(f"Learning rate: {LEARNING_RATE}")
print(f"Class weights: Real={class_weight_dict[0]:.3f}, AI={class_weight_dict[1]:.3f}")

In [ ]:
# ─── Trening ───
EPOCHS = 15

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    steps_per_epoch=STEPS_PER_EPOCH,
    validation_steps=VALIDATION_STEPS,
    class_weight=class_weight_dict,
    callbacks=my_callbacks,
    verbose=1
)

In [ ]:
def plot_training(history, title=""):
    metrics = [
        ('loss', 'val_loss', 'Loss'),
        ('accuracy', 'val_accuracy', 'Accuracy'),
        ('auc', 'val_auc', 'AUC-ROC'),
        ('precision', 'val_precision', 'Precision'),
        ('recall', 'val_recall', 'Recall'),
    ]
    # Filtruj metryki które istnieją w historii
    metrics = [(t,v,n) for t,v,n in metrics if t in history.history]

    fig, axes = plt.subplots(1, len(metrics), figsize=(5*len(metrics), 4))
    if len(metrics) == 1:
        axes = [axes]

    for ax, (train_key, val_key, name) in zip(axes, metrics):
        ax.plot(history.history[train_key], label='Train', linewidth=2)
        if val_key in history.history:
            ax.plot(history.history[val_key], label='Val', linewidth=2)
        ax.set_title(name, fontweight='bold')
        ax.set_xlabel('Epoch')
        ax.legend()
        ax.grid(True, alpha=0.3)

        # Dodaj oś referencyjną na 0.5 dla metryk [0,1]
        if name != 'Loss':
            ax.axhline(y=0.5, color='gray', linestyle=':', alpha=0.5)

    plt.suptitle(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

    # Najlepsze wyniki
    if 'val_auc' in history.history:
        best_epoch = np.argmax(history.history['val_auc'])
        print(f"\nNajlepsza epoka: {best_epoch + 1}")
        print(f"  Val AUC:      {history.history['val_auc'][best_epoch]:.4f}")
        print(f"  Val Accuracy: {history.history['val_accuracy'][best_epoch]:.4f}")
        print(f"  Val Loss:     {history.history['val_loss'][best_epoch]:.4f}")

plot_training(history, "Trening — Fine-tuning od warstwy 100")

## 4. Zapis modelu

In [ ]:
# Zapis finalnego modelu
model.save(FINAL_MODEL_PATH)
print(f"✅ Model:      {FINAL_MODEL_PATH}")

model.save_weights(WEIGHTS_PATH)
print(f"✅ Wagi:       {WEIGHTS_PATH}")

print(f"✅ Checkpoint: {CHECKPOINT_PATH}")

for path in [FINAL_MODEL_PATH, WEIGHTS_PATH, CHECKPOINT_PATH]:
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / (1024 * 1024)
        print(f"   {path}: {size_mb:.1f} MB")

## 5. Ewaluacja

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

val_labels = []
val_preds = []

for images, labels in val_ds.take(VALIDATION_STEPS):
    preds = model.predict(images, verbose=0)
    val_preds.extend(preds.flatten())
    val_labels.extend(labels.numpy())

val_labels = np.array(val_labels)
val_preds = np.array(val_preds)
val_pred_classes = (val_preds > 0.5).astype(int)

print("=== Classification Report ===")
print(classification_report(
    val_labels, val_pred_classes,
    target_names=['Real', 'AI-Generated']
))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

cm = confusion_matrix(val_labels, val_pred_classes)
ConfusionMatrixDisplay(cm, display_labels=['Real', 'AI']).plot(ax=ax1, cmap='Blues')
ax1.set_title("Confusion Matrix", fontweight='bold')

cm_norm = confusion_matrix(val_labels, val_pred_classes, normalize='true')
ConfusionMatrixDisplay(cm_norm, display_labels=['Real', 'AI']).plot(ax=ax2, cmap='Greens')
ax2.set_title("Normalized", fontweight='bold')

plt.tight_layout()
plt.show()

# Histogram pewności predykcji
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(val_preds[val_labels == 0], bins=50, alpha=0.6, label='Real', color='green')
ax.hist(val_preds[val_labels == 1], bins=50, alpha=0.6, label='AI-Generated', color='red')
ax.axvline(x=0.5, color='black', linestyle='--', label='Próg 0.5')
ax.set_xlabel('P(AI-Generated)')
ax.set_ylabel('Liczba obrazów')
ax.set_title('Rozkład pewności predykcji — separacja klas', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

## Troubleshooting

Jeśli model nadal nie uczy się (AUC < 0.6 po 5 epokach):

**1. Odmroź więcej warstw:**
```python
model = build_model(unfreeze_from=50)  # 80% odmrożone
```

**2. Odmroź WSZYSTKO (pełny fine-tuning):**
```python
model = build_model(unfreeze_from=0)
# Użyj mniejszego LR: 5e-5
```

**3. Sprawdź czy GPU jest używane:**
```python
print(tf.config.list_physical_devices('GPU'))
# Jeśli puste → CPU, trening będzie wolny ale powinien działać
```

**4. Zmniejsz batch size (jeśli OOM na GPU):**
```python
BATCH_SIZE = 16
```

**5. Spróbuj inny backbone (ResNet50 — prostszy):**
```python
from tensorflow.keras.applications import ResNet50
base = ResNet50(include_top=False, weights='imagenet', input_shape=(224,224,3))
# ResNet50 NIE ma wbudowanej normalizacji!
# Użyj: tf.keras.applications.resnet50.preprocess_input
```


## Następne kroki

1. **Multi-class (Label_B)** — zmień Dense(1, sigmoid) → Dense(6, softmax) + sparse_categorical_crossentropy
2. **Frequency features** — dodaj FFT jako dodatkowe kanały
3. **Ensemble** — połącz EfficientNet + ViT
4. **Grad-CAM** — wizualizacja "na co patrzy model"
5. **Cross-generator testing** — czy model generalizuje na niewidziane generatory?
